# الفصل العشرون: مسار تحليل البيانات (pandas, numpy, matplotlib)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/chapter-20.ipynb)

> **الوقت المتوقع:** ~11 دقيقة قراءة · **11** قسمًا · **36** مثالًا برمجيًا.

**كيف تستخدم هذا الدفتر؟**

1. شغّل الخلايا بالترتيب من الأعلى إلى الأسفل (`Shift` + `Enter`).
2. خلايا الشيفرة قابلة للتعديل: غيّر القيم وأعد التشغيل لتجرّب.
3. خلايا 🧾 معروضة للقراءة فقط (أوامر طرفية، أو إدخال يدوي، أو مقتطف ناقص)، وخلايا ⚠️ تُظهر خطأً مقصودًا يُلتقط تلقائيًا فلا يتوقف «تشغيل الكل».

> **Kaggle:** نزّل الملف من المستودع ثم `File ▸ Import Notebook`، أو استورده مباشرة من GitHub بعد ربط حسابك.

<details>
<summary><b>محتويات الفصل</b></summary>

- **20.1** لماذا تحليل البيانات؟
- **20.2** تثبيت المكتبات
- **20.3** NumPy: أساس الحوسبة الرقمية
- **20.4** pandas: العمود الفقري لتحليل البيانات
- **20.5** Matplotlib: تحويل الأرقام إلى رسوم
- **20.6** مشروع تطبيقي متكامل: تحليل مبيعات متجر إلكتروني
- **20.7** أخطاء شائعة في تحليل البيانات
- **20.8** تمارين
- **20.9** اختبر فهمك (Quiz)
- **20.10** ملخص الفصل (Cheat Sheet)
- **20.11** ماذا بعد؟

</details>

In [ ]:
#@title ⚙️ تهيئة البيئة  { display-mode: "form" }
import sys, os, random, pathlib

print("Python:", sys.version.split()[0], "|", sys.platform)
assert sys.version_info >= (3, 10), "هذا الكتاب يفترض Python 3.10 أو أحدث"

SEED = 42
random.seed(SEED)

# تثبيت الحزم الناقصة فقط (سريع إن كانت مثبّتة أصلًا)
import importlib.util as _iu
_need = [m for m in ["pandas", "numpy", "matplotlib", "openpyxl", "arabic_reshaper", "bidi"] if _iu.find_spec(m) is None]
if _need:
    %pip install -q pandas numpy matplotlib openpyxl arabic-reshaper python-bidi

# مجلد عمل معزول حتى لا تتناثر الملفات في جذر الدفتر
WORK = pathlib.Path("book_runtime")
WORK.mkdir(exist_ok=True)
os.chdir(WORK)
print("مجلد العمل:", pathlib.Path.cwd().name)

# مجلدات الإخراج التي تكتب فيها الأمثلة
pathlib.Path('data').mkdir(exist_ok=True)

# إعداد خاص بهذا الفصل
import pandas as _pd
_sales = _pd.DataFrame({
    'المنتج': ['لابتوب', 'ماوس', 'شاشة', 'لابتوب', 'ماوس'],
    'المدينة': ['الرياض', 'جدة', 'الرياض', 'جدة', 'الرياض'],
    'السعر': [4500, 120, 900, 4700, 110],
    'الكمية': [2, 10, 3, 1, 8],
})
if not pathlib.Path('sales.csv').exists():
    _sales.to_csv('sales.csv', index=False, encoding='utf-8')
    _sales.to_json('sales.json', orient='records', force_ascii=False)
    _sales.to_excel('sales.xlsx', index=False, sheet_name='2024')
    print('أُنشئت ملفات المبيعات النموذجية: sales.csv / sales.json / sales.xlsx')

# --- عرض العربية في رسوم Matplotlib بشكل متصل وصحيح الاتجاه ---
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
try:
    import arabic_reshaper
    from bidi.algorithm import get_display
    _shape = lambda s: get_display(arabic_reshaper.reshape(s))
    import matplotlib.text as _mt
    _orig = _mt.Text.set_text
    def _set_text(self, s):
        if isinstance(s, str) and any('\u0600' <= c <= '\u06FF' for c in s):
            s = _shape(s)
        return _orig(self, s)
    _mt.Text.set_text = _set_text
    print("دعم العربية في الرسوم: مُفعّل")
except ImportError:
    print("تنبيه: ثبّت arabic-reshaper وpython-bidi لعرض العربية في الرسوم")

> ℹ️ يُضبط Matplotlib تلقائيًا لعرض العناوين العربية متصلةً وصحيحة الاتجاه.

## 20.1 لماذا تحليل البيانات؟

تخيّل أنك تملك ملف Excel يحتوي على **500,000 صف** من بيانات المبيعات، وتريد معرفة: ما هو المنتج الأكثر مبيعًا في كل مدينة خلال الربع الأخير؟ ما متوسط قيمة الفاتورة لكل عميل؟ هل هناك اتجاه تصاعدي أو تنازلي في المبيعات؟

في Excel، ستحتاج إلى معادلات معقدة، جداول محورية (Pivot Tables)، وربما ساعات من النقر والسحب — وقد ينهار البرنامج أصلاً مع هذا الحجم من البيانات. في Python، يمكنك الإجابة على هذه الأسئلة في **أسطر معدودة**، بسرعة فائقة، وبطريقة **قابلة لإعادة الاستخدام**: نفس الكود يعمل غدًا على بيانات جديدة دون أي تعديل.

هذا هو سبب كون **تحليل البيانات (Data Analysis)** أحد أقوى تطبيقات Python في العالم الحقيقي. شركات مثل Netflix وSpotify وAmazon تستخدم بالضبط المكتبات التي ستتعلمها في هذا الفصل لاتخاذ قرارات تجارية تساوي ملايين الدولارات.

> **🔄 مقارنة: Excel مقابل Python لتحليل البيانات**
>
> | المعيار | Excel | Python (pandas) |
> |---|---|---|
> | حجم البيانات | يتعثر مع أكثر من ~1 مليون صف | يتعامل بسهولة مع ملايين الصفوف |
> | التكرار | يدوي، عرضة للخطأ البشري | كود يُعاد تشغيله تلقائيًا |
> | التوثيق | صعب تتبع «ماذا فعلت بالضبط؟» | الكود نفسه هو التوثيق |
> | التكامل | محدود | يتكامل مع قواعد البيانات، APIs، الذكاء الاصطناعي |
> | التعلم الآلي | غير ممكن مباشرة | يتكامل مباشرة مع scikit-learn, TensorFlow |
> | الرسوم البيانية | جيدة للبساطة | أقوى وأكثر تخصيصًا (matplotlib, seaborn, plotly) |
>
> **الخلاصة:** Excel ممتاز للمهام السريعة والصغيرة. لكن عندما تكبر البيانات أو تتكرر المهمة، Python هو الخيار الاحترافي.

في هذا الفصل، سنتعلم ثلاث مكتبات تُشكّل «الثالوث المقدس» لتحليل البيانات في Python:

1. **NumPy**: الأساس الرياضي — يتعامل مع المصفوفات (Arrays) والعمليات الحسابية بسرعة فائقة.
2. **pandas**: الأداة الرئيسية — تتعامل مع البيانات الجدولية (صفوف وأعمدة) كما في Excel، لكن ببرمجة.
3. **Matplotlib**: التصور — تحويل الأرقام إلى رسوم بيانية مفهومة.

ثم سنطبق كل هذا في **مشروع متكامل لتحليل بيانات مبيعات حقيقية**.

## 20.2 تثبيت المكتبات

كما تعلمنا في الفصل 18، نستخدم `uv` لإدارة التبعيات:

```bash
uv add numpy pandas matplotlib
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

هذا الأمر سيحمّل ويثبّت المكتبات الثلاث، ويضيفها إلى `pyproject.toml`.

> **💡 نصيحة خبير:** غالبًا تُستخدم أسماء مستعارة (aliases) قياسية ومتعارف عليها عالميًا عند استيراد هذه المكتبات. التزم بها دائمًا — فكل مطوّر بيانات في العالم يتوقعها:
> ```python
> import numpy as np
> import pandas as pd
> import matplotlib.pyplot as plt
> ```

## 20.3 NumPy: أساس الحوسبة الرقمية

### لماذا لا نستخدم القوائم العادية؟

في الفصل 8، تعلمنا القوائم (`list`). لماذا نحتاج شيئًا آخر؟

In [ ]:
# محاولة جمع قائمتين من الأرقام عنصرًا بعنصر
prices = [10, 20, 30]
discounts = [1, 2, 3]

# هذا لن يعمل كما تتوقع!
result = prices + discounts
print(result)  # [10, 20, 30, 1, 2, 3] -- دمج وليس جمع!

القوائم العادية لا تدعم العمليات الرياضية «الطبيعية» بين عناصرها. لحل هذه المشكلة، نحتاج حلقة `for`:

In [ ]:
result = [p - d for p, d in zip(prices, discounts)]
print(result)  # [9, 18, 27]

هذا يعمل، لكنه **بطيء جدًا** مع ملايين العناصر، لأن Python يُنفّذ الحلقة عنصرًا تلو الآخر بشكل مفسَّر (interpreted).

**NumPy** يحل هذه المشكلة عبر هيكل بيانات جديد يُسمى `ndarray` (N-dimensional array)، مُحسَّن داخليًا بلغة C، يسمح بإجراء العمليات الحسابية على **كل العناصر دفعة واحدة** (Vectorization) بسرعة تفوق القوائم العادية بعشرات أو مئات المرات.

In [ ]:
import numpy as np

prices = np.array([10, 20, 30])
discounts = np.array([1, 2, 3])

result = prices - discounts
print(result)  # [9 18 27]
print(type(result))  # <class 'numpy.ndarray'>

> **🔄 مقارنة: list مقابل ndarray**
>
> | الخاصية | `list` | `numpy.ndarray` |
> |---|---|---|
> | أنواع العناصر | مختلطة (int, str, ...) | نوع واحد موحّد (كلها int أو كلها float...) |
> | العمليات الحسابية | تحتاج حلقات | مباشرة (Vectorized) |
> | السرعة | أبطأ | أسرع بكثير (مُنفّذة بلغة C) |
> | استهلاك الذاكرة | أكبر | أقل بكثير |
> | الاستخدام | عام (أي بيانات) | حسابات رقمية/علمية |

### إنشاء المصفوفات

In [ ]:
import numpy as np

# من قائمة
a = np.array([1, 2, 3, 4, 5])
print(a)          # [1 2 3 4 5]
print(a.dtype)    # int64 (نوع البيانات الموحّد)

# مصفوفة ثنائية الأبعاد (مصفوفة من مصفوفات = جدول)
matrix = np.array([
    [1, 2, 3],
    [4, 5, 6],
])
print(matrix.shape)  # (2, 3) -- صفّان، 3 أعمدة

# دوال إنشاء جاهزة
zeros = np.zeros(5)           # [0. 0. 0. 0. 0.]
ones = np.ones((2, 3))        # مصفوفة 2x3 من الآحاد
range_arr = np.arange(0, 10, 2)   # [0 2 4 6 8]  (مثل range لكن تُرجع array)
linspace = np.linspace(0, 1, 5)   # [0.   0.25 0.5  0.75 1.  ] -- 5 قيم متباعدة بالتساوي
random_arr = np.random.rand(3)    # 3 أرقام عشوائية بين 0 و1

### العمليات الحسابية (Vectorization)

هذه هي القوة الحقيقية لـ NumPy — لاحظ عدم وجود أي حلقة `for`:

In [ ]:
prices = np.array([100, 200, 300, 400])

# عملية على كل عنصر دفعة واحدة
with_tax = prices * 1.15
print(with_tax)  # [115. 230. 345. 460.]

# مقارنة تُرجع مصفوفة من Boolean
expensive = prices > 250
print(expensive)  # [False False  True  True]

# استخدام هذه المصفوفة Boolean للتصفية! (Boolean Indexing)
print(prices[expensive])  # [300 400]

# أو في سطر واحد
print(prices[prices > 250])  # [300 400]

### إحصائيات أساسية

In [ ]:
sales = np.array([120, 340, 210, 450, 90, 380])

print(sales.sum())     # 1590 -- المجموع
print(sales.mean())    # 265.0 -- المتوسط
print(sales.max())     # 450 -- الأكبر
print(sales.min())     # 90 -- الأصغر
print(sales.std())     # الانحراف المعياري
print(sales.argmax())  # 3 -- فهرس العنصر الأكبر
print(np.median(sales))  # الوسيط

> **⚠️ تحذير: Broadcasting قد يُربك المبتدئين**
>
> عندما تجري عملية بين مصفوفة وعدد مفرد (scalar)، يطبّق NumPy العملية على كل عنصر تلقائيًا — هذا يُسمى **Broadcasting**:
> ```python
> arr = np.array([1, 2, 3])
> print(arr + 10)  # [11 12 13] -- وليس خطأ!
> ```
> هذا سلوك **مقصود ومفيد جدًا**، لكن يجب أن تفهمه حتى لا يفاجئك.

> **💡 نصيحة خبير:** NumPy هو الأساس الذي تُبنى عليه pandas وscikit-learn وTensorFlow. لن تحتاج غالبًا لاستخدامه مباشرة بشكل مكثف في تحليل البيانات اليومي (ستستخدم pandas بدلاً منه)، لكن **فهمه ضروري** لأن pandas مبني عليه بالكامل، وستقابله باستمرار.

## 20.4 pandas: العمود الفقري لتحليل البيانات

إذا كان NumPy يتعامل مع أرقام في مصفوفات، فإن **pandas** يتعامل مع بيانات **جدولية** حقيقية: صفوف وأعمدة لها أسماء، أنواع بيانات مختلفة (نصوص، أرقام، تواريخ)، وقيم مفقودة — تمامًا كملف Excel أو جدول في قاعدة بيانات.

### هيكلا البيانات الأساسيان

**1. `Series`**: عمود واحد من البيانات (كـ«قائمة» لكن أذكى، بفهارس مسمّاة).

In [ ]:
import pandas as pd

temperatures = pd.Series([22, 25, 19, 30], index=["الأحد", "الاثنين", "الثلاثاء", "الأربعاء"])
print(temperatures)

```
الأحد       22
الاثنين     25
الثلاثاء    19
الأربعاء    30
dtype: int64
```

**2. `DataFrame`**: جدول كامل — مجموعة من `Series` تشترك في نفس الفهرس. هذا هو الهيكل الذي ستستخدمه 95% من الوقت.

In [ ]:
data = {
    "اسم المنتج": ["لابتوب", "ماوس", "لوحة مفاتيح", "شاشة"],
    "السعر": [3500, 50, 120, 900],
    "الكمية": [10, 50, 30, 15],
}

df = pd.DataFrame(data)
print(df)

```
  اسم المنتج  السعر  الكمية
0      لابتوب   3500      10
1        ماوس     50      50
2   لوحة مفاتيح    120      30
3        شاشة    900      15
```

> **🔄 مقارنة: DataFrame مقابل ورقة Excel**
>
> | Excel | pandas |
> |---|---|
> | ورقة عمل (Sheet) | `DataFrame` |
> | عمود | `Series` |
> | رقم الصف | `index` |
> | اسم العمود | `columns` |
> | جدول محوري (Pivot) | `.groupby()` + `.pivot_table()` |
> | فلتر/تصفية | Boolean Indexing |
> | VLOOKUP | `.merge()` |

### قراءة البيانات من ملفات

نادرًا ما نُنشئ DataFrame يدويًا؛ عادة نقرأه من ملف:

In [ ]:
# من CSV
df = pd.read_csv("sales.csv")

# من Excel (يتطلب: uv add openpyxl)
df = pd.read_excel("sales.xlsx", sheet_name="2024")

# من JSON
df = pd.read_json("sales.json")

# تحديد الترميز إن كانت البيانات عربية
df = pd.read_csv("sales.csv", encoding="utf-8")

> **⚠️ خطأ شائع:** نسيان `encoding="utf-8"` عند قراءة ملفات تحتوي نصوصًا عربية قد يؤدي إلى ظهور رموز غريبة (Mojibake) أو خطأ `UnicodeDecodeError`. إذا واجهت مشكلة، جرّب `encoding="utf-8-sig"` (خصوصًا للملفات المُصدَّرة من Excel على Windows).

### استكشاف البيانات: أول خطوة دائمًا

بمجرد تحميل أي بيانات، نبدأ دائمًا بهذه الأوامر الاستكشافية:

In [ ]:
df.head()       # أول 5 صفوف (الافتراضي)
df.head(10)     # أول 10 صفوف
df.tail(3)      # آخر 3 صفوف

df.info()       # معلومات عامة: الأعمدة، الأنواع، القيم غير الفارغة
df.describe()   # إحصائيات (متوسط، انحراف معياري، أدنى، أقصى...) للأعمدة الرقمية

df.shape        # (عدد الصفوف, عدد الأعمدة) -- خاصية وليست دالة!
df.columns      # أسماء الأعمدة
df.dtypes       # نوع بيانات كل عمود
df.index        # معلومات الفهرس

مثال عملي:

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "المنتج": ["لابتوب", "ماوس", "شاشة", "لابتوب", "ماوس"],
    "السعر": [3500, 50, 900, 3600, 55],
    "الكمية": [2, 10, 3, 1, 8],
    "المدينة": ["الرياض", "جدة", "الرياض", "الدمام", "جدة"],
})

print(df.info())

```
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   المنتج  5 non-null      object
 1   السعر   5 non-null      int64
 2   الكمية  5 non-null      int64
 3   المدينة 5 non-null      object
dtypes: int64(2), object(2)
```

🧾 **للقراءة فقط** — مقتطف غير مكتمل.

### الفهرسة والاختيار: `loc` و `iloc`

هذا أهم مفهوم عليك إتقانه في pandas.

In [ ]:
# اختيار عمود واحد -> يُرجع Series
print(df["المنتج"])

# اختيار عدة أعمدة -> يُرجع DataFrame (لاحظ قائمة داخل قوسين)
print(df[["المنتج", "السعر"]])

# loc: الاختيار بالاسم/التسمية
print(df.loc[0])              # الصف الأول (حسب التسمية، وهنا تتطابق مع الرقم)
print(df.loc[0, "المنتج"])    # قيمة محددة
print(df.loc[0:2, "السعر"])   # نطاق من الصفوف، عمود محدد

# iloc: الاختيار بالموضع الرقمي (مثل القوائم تمامًا)
print(df.iloc[0])        # أول صف بغض النظر عن تسميته
print(df.iloc[0, 1])     # الصف 0، العمود 1
print(df.iloc[-1])       # آخر صف

> **🔄 مقارنة: `loc` مقابل `iloc`**
>
> | | `loc` | `iloc` |
> |---|---|---|
> | يعتمد على | **التسمية (label)** | **الموضع (position)** |
> | مثال | `df.loc["a"]` | `df.iloc[0]` |
> | النطاقات | شاملة للطرفين `0:2` تشمل 2 | غير شاملة للطرف الأخير، مثل `range` |

### التصفية (Filtering) بـ Boolean Indexing

تمامًا كما في NumPy:

In [ ]:
# كل المنتجات التي سعرها أكبر من 500
expensive = df[df["السعر"] > 500]
print(expensive)

# شروط متعددة: استخدم & و | (وليس and/or) مع أقواس إلزامية
cheap_riyadh = df[(df["السعر"] < 1000) & (df["المدينة"] == "الرياض")]
print(cheap_riyadh)

# التحقق من انتماء القيمة لقائمة
target_cities = df[df["المدينة"].isin(["جدة", "الدمام"])]

# استبعاد قيم
not_riyadh = df[df["المدينة"] != "الرياض"]

> **⚠️ تحذير: `and`/`or` لا تعمل مع pandas!**
> ```python
> # ❌ خطأ -- سيرفع ValueError
> df[(df[«السعر»] > 100) and (df[«الكمية»] > 5)]
>
> # ✅ صحيح -- استخدم & و | مع أقواس حول كل شرط
> df[(df[«السعر»] > 100) & (df[«الكمية»] > 5)]
> ```
> السبب: `and`/`or` في Python تتوقع قيمة منطقية واحدة (True/False)، بينما `(df["السعر"] > 100)` تُرجع `Series` كاملة من قيم Boolean، وليس قيمة واحدة. عوامل `&` و`|` مصمّمة للعمل عنصرًا بعنصر.

### إضافة وتعديل وحذف الأعمدة

In [ ]:
# إضافة عمود جديد محسوب
df["الإجمالي"] = df["السعر"] * df["الكمية"]
print(df)

# إضافة عمود بشرط باستخدام numpy.where
df["الفئة"] = np.where(df["السعر"] > 500, "غالي", "رخيص")

# تعديل عمود موجود
df["السعر"] = df["السعر"] * 1.15  # إضافة ضريبة 15%

# حذف عمود
df = df.drop(columns=["الفئة"])

# إعادة تسمية الأعمدة
df = df.rename(columns={"السعر": "سعر_الوحدة"})

### التعامل مع القيم المفقودة (Missing Data)

البيانات الحقيقية دائمًا «فوضوية» — تحتوي على قيم مفقودة تُمثَّل في pandas بـ `NaN` (Not a Number).

In [ ]:
data = {
    "اسم": ["أحمد", "سارة", None, "خالد"],
    "العمر": [25, None, 30, 28],
}
df = pd.DataFrame(data)

print(df.isna())          # جدول Boolean يوضح أماكن القيم المفقودة
print(df.isna().sum())    # عدد القيم المفقودة في كل عمود

# حذف الصفوف التي تحتوي أي قيمة مفقودة
clean_df = df.dropna()

# حذف فقط إذا كان عمود معين فارغًا
clean_df = df.dropna(subset=["اسم"])

# تعويض القيم المفقودة بقيمة محددة
df["العمر"] = df["العمر"].fillna(df["العمر"].mean())  # بالمتوسط
df["اسم"] = df["اسم"].fillna("غير معروف")

> **💡 نصيحة خبير:** القرار بين حذف القيم المفقودة (`dropna`) أو تعويضها (`fillna`) يعتمد على السياق. إذا كانت نسبة القيم المفقودة صغيرة جدًا (أقل من 5%)، الحذف غالبًا آمن. إذا كانت كبيرة، التعويض الذكي (بالمتوسط، الوسيط، أو قيمة منطقية) أفضل لتجنب فقدان بيانات ثمينة.

### الفرز (Sorting)

In [ ]:
# 🔄 نعيد تعريف df ببيانات المبيعات حتى يعمل المثال كما في الكتاب
df = pd.DataFrame({
    'المنتج': ['لابتوب', 'ماوس', 'شاشة', 'لابتوب', 'ماوس'],
    'المدينة': ['الرياض', 'جدة', 'الرياض', 'جدة', 'الرياض'],
    'السعر': [4500, 120, 900, 4700, 110],
    'الكمية': [2, 10, 3, 1, 8],
})

In [ ]:
df_sorted = df.sort_values("السعر")                    # تصاعدي
df_sorted = df.sort_values("السعر", ascending=False)    # تنازلي
df_sorted = df.sort_values(["المدينة", "السعر"])         # فرز متعدد المستويات

### التجميع: `groupby` — أقوى أداة في pandas

`groupby` يُجيب على أسئلة مثل: «ما مجموع المبيعات **لكل** مدينة؟»

In [ ]:
df = pd.DataFrame({
    "المنتج": ["لابتوب", "ماوس", "شاشة", "لابتوب", "ماوس", "شاشة"],
    "المدينة": ["الرياض", "جدة", "الرياض", "جدة", "الرياض", "جدة"],
    "الإجمالي": [7000, 500, 2700, 3600, 450, 1800],
})

# مجموع المبيعات لكل مدينة
by_city = df.groupby("المدينة")["الإجمالي"].sum()
print(by_city)

```
المدينة
الرياض    10150
جدة        5900
Name: الإجمالي, dtype: int64
```

In [ ]:
# عدة إحصائيات دفعة واحدة
summary = df.groupby("المدينة")["الإجمالي"].agg(["sum", "mean", "count", "max"])
print(summary)

# التجميع حسب أكثر من عمود
by_city_product = df.groupby(["المدينة", "المنتج"])["الإجمالي"].sum()
print(by_city_product)

# تحويل النتيجة إلى DataFrame عادي مرة أخرى
result = df.groupby("المدينة")["الإجمالي"].sum().reset_index()

> **🔄 مقارنة: `groupby` في pandas مقابل `GROUP BY` في SQL**
>
> ```sql
> SELECT المدينة, SUM(الإجمالي)
> FROM sales
> GROUP BY المدينة;
> ```
> ```python
> df.groupby(«المدينة»)[«الإجمالي»].sum()
> ```
> الفكرة متطابقة تمامًا! إذا كنت تعرف SQL، فهم `groupby` سيكون سهلاً جدًا.

### الجداول المحورية: `pivot_table`

In [ ]:
pivot = df.pivot_table(
    values="الإجمالي",
    index="المدينة",
    columns="المنتج",
    aggfunc="sum",
    fill_value=0,
)
print(pivot)

```
المنتج    لابتوب  ماوس  شاشة
المدينة
الرياض     7000   450   2700
جدة        3600   500   1800
```

تمامًا كجدول Pivot في Excel، لكن بسطر واحد وقابل لإعادة الاستخدام على أي بيانات.

### دمج البيانات: `merge` و `concat`

In [ ]:
customers = pd.DataFrame({
    "customer_id": [1, 2, 3],
    "اسم": ["أحمد", "سارة", "خالد"],
})

orders = pd.DataFrame({
    "order_id": [101, 102, 103, 104],
    "customer_id": [1, 2, 1, 3],
    "المبلغ": [200, 450, 120, 300],
})

# الدمج (مثل JOIN في SQL)
merged = pd.merge(orders, customers, on="customer_id", how="left")
print(merged)

# دمج/تكديس DataFrames فوق بعضها (مثل إلحاق ملفات شهر بشهر)
jan_sales = pd.DataFrame({"المنتج": ["لابتوب"], "الإجمالي": [3500]})
feb_sales = pd.DataFrame({"المنتج": ["ماوس"], "الإجمالي": [50]})
all_sales = pd.concat([jan_sales, feb_sales], ignore_index=True)

> **جدول مرجعي: أنواع `how` في `merge`**
>
> | النوع | الوصف |
> |---|---|
> | `"inner"` (افتراضي) | فقط الصفوف المتطابقة في الجدولين |
> | `"left"` | كل صفوف الجدول الأيسر + المطابقة من الأيمن |
> | `"right"` | كل صفوف الجدول الأيمن + المطابقة من الأيسر |
> | `"outer"` | كل الصفوف من الجدولين (مع `NaN` حيث لا تطابق) |

### التعامل مع التواريخ

In [ ]:
df = pd.DataFrame({
    "التاريخ": ["2024-01-15", "2024-02-20", "2024-03-10"],
    "المبيعات": [1000, 1500, 1200],
})

# تحويل النص إلى تاريخ حقيقي
df["التاريخ"] = pd.to_datetime(df["التاريخ"])

# استخراج معلومات من التاريخ
df["الشهر"] = df["التاريخ"].dt.month
df["اسم_اليوم"] = df["التاريخ"].dt.day_name()

print(df)

### تصدير البيانات

In [ ]:
df.to_csv("output.csv", index=False, encoding="utf-8-sig")
df.to_excel("output.xlsx", index=False)
df.to_json("output.json", orient="records", force_ascii=False, indent=2)

> **💡 نصيحة خبير:** استخدم دائمًا `index=False` عند التصدير إلى CSV/Excel، وإلا ستحصل على عمود إضافي غير مرغوب فيه يحتوي أرقام الفهرس الداخلية لـ pandas.

## 20.5 Matplotlib: تحويل الأرقام إلى رسوم

الأرقام في جدول صعبة الفهم بصريًا. رسم بياني واحد قد يوضح اتجاهًا يستغرق فهمه من الجدول دقائق طويلة.

### الرسم الأساسي

In [ ]:
import matplotlib.pyplot as plt

months = ["يناير", "فبراير", "مارس", "أبريل", "مايو"]
sales = [1200, 1900, 1500, 2200, 2800]

plt.plot(months, sales)
plt.title("المبيعات الشهرية")
plt.xlabel("الشهر")
plt.ylabel("المبيعات (ريال)")
plt.show()  # يفتح نافذة لعرض الرسم

> **⚠️ تحذير: دعم اللغة العربية في matplotlib**
> matplotlib لا يدعم اتجاه الكتابة من اليمين لليسار (RTL) أو ربط الحروف العربية بشكل جيد افتراضيًا — قد تظهر الحروف منفصلة أو معكوسة. للحصول على نص عربي صحيح، استخدم مكتبة `arabic-reshaper` مع `python-bidi`:
> ```python
> import arabic_reshaper
> from bidi.algorithm import get_display
>
> def fix_arabic(text: str) -> str:
>     reshaped = arabic_reshaper.reshape(text)
>     return get_display(reshaped)
>
> plt.title(fix_arabic(«المبيعات الشهرية»))
> ```
> في الأمثلة التالية سنستخدم أحيانًا تسميات إنجليزية للتبسيط، لكن اعتمد هذا الحل في مشاريعك الحقيقية عند الحاجة لنصوص عربية في الرسوم.

### أنواع الرسوم الشائعة

In [ ]:
import matplotlib.pyplot as plt

categories = ["لابتوب", "ماوس", "شاشة", "لوحة مفاتيح"]
values = [45, 120, 30, 80]

# رسم بياني خطي -- مناسب للاتجاهات عبر الزمن
plt.plot(categories, values, marker="o")
plt.show()

# رسم بياني شريطي (Bar Chart) -- مناسب للمقارنة بين فئات
plt.bar(categories, values, color="steelblue")
plt.show()

# رسم دائري (Pie Chart) -- مناسب لإظهار النسب من الكل
plt.pie(values, labels=categories, autopct="%1.1f%%")
plt.show()

# مخطط التشتت (Scatter Plot) -- مناسب لإظهار العلاقة بين متغيرين
prices = [3500, 50, 900, 120]
plt.scatter(prices, values)
plt.xlabel("السعر")
plt.ylabel("الكمية المباعة")
plt.show()

# المدرج التكراري (Histogram) -- مناسب لتوزيع البيانات
import numpy as np
ages = np.random.normal(30, 5, 1000)  # 1000 عمر عشوائي حول متوسط 30
plt.hist(ages, bins=20, color="coral", edgecolor="black")
plt.show()

> **جدول مرجعي: أي رسم تختار؟**
>
> | نوع البيانات | السؤال | الرسم المناسب |
> |---|---|---|
> | تطور عبر الزمن | هل المبيعات تزداد؟ | `plt.plot()` (خطي) |
> | مقارنة فئات | أي منتج الأكثر مبيعًا؟ | `plt.bar()` (شريطي) |
> | نسب من كل | ما حصة كل قسم من الميزانية؟ | `plt.pie()` (دائري) |
> | علاقة بين متغيرين | هل السعر يؤثر على الكمية المباعة؟ | `plt.scatter()` (تشتت) |
> | توزيع قيم | كيف تتوزع أعمار العملاء؟ | `plt.hist()` (مدرج تكراري) |

### تخصيص الرسوم

In [ ]:
plt.figure(figsize=(10, 6))  # حجم الرسم (عرض, ارتفاع) بالبوصة

plt.plot(months, sales, marker="o", linestyle="--", color="green", linewidth=2, label="2024")
plt.title("المبيعات الشهرية", fontsize=16, fontweight="bold")
plt.xlabel("الشهر")
plt.ylabel("المبيعات (ريال)")
plt.legend()        # إظهار الـ label الموجود أعلاه
plt.grid(True, alpha=0.3)   # شبكة خفيفة في الخلفية
plt.tight_layout()  # ترتيب تلقائي لمنع تداخل العناصر

plt.savefig("sales_chart.png", dpi=300)  # حفظ كصورة (بدلاً من/بالإضافة إلى show)
plt.show()

### رسوم متعددة في شكل واحد: `subplots`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))  # صف واحد، عمودان

axes[0].plot(months, sales, marker="o")
axes[0].set_title("خطي")

axes[1].bar(months, sales, color="orange")
axes[1].set_title("شريطي")

plt.tight_layout()
plt.show()

### الرسم مباشرة من pandas

ميزة مريحة جدًا: DataFrame وSeries لديهما دالة `.plot()` مدمجة تستخدم matplotlib خلف الكواليس:

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "الشهر": ["يناير", "فبراير", "مارس"],
    "المبيعات": [1200, 1900, 1500],
})

df.plot(x="الشهر", y="المبيعات", kind="bar", title="المبيعات الشهرية")
plt.show()

# أو مباشرة من نتيجة groupby
df.groupby("الشهر")["المبيعات"].sum().plot(kind="pie", autopct="%1.1f%%")
plt.show()

> **💡 نصيحة خبير:** للتحليل الاستكشافي السريع (أنت فقط تستكشف البيانات)، استخدم `.plot()` من pandas — أسرع وأقل كتابة. للرسوم النهائية التي ستُعرض أو تُنشر في تقرير، استخدم matplotlib مباشرة للحصول على تحكم كامل في التفاصيل.

> **🔄 مقارنة: matplotlib مقابل مكتبات تصور أخرى**
>
> | المكتبة | الاستخدام الأمثل |
> |---|---|
> | **matplotlib** | الأساس، تحكم كامل، لكن أكثر كتابة للتفاصيل الجميلة |
> | **seaborn** | مبنية فوق matplotlib، رسوم إحصائية جاهزة وأجمل افتراضيًا |
> | **plotly** | رسوم تفاعلية (تكبير، تمرير الفأرة) مناسبة لتطبيقات الويب |
>
> كمبتدئ، أتقن matplotlib أولاً — فهمها يجعل تعلم الباقي أسهل بكثير لأنها الأساس الذي تُبنى عليه.

## 20.6 مشروع تطبيقي متكامل: تحليل مبيعات متجر إلكتروني

سنبني الآن سكربت تحليل بيانات متكامل لمتجر إلكتروني وهمي، يغطي: تحميل البيانات، تنظيفها، تحليلها من زوايا متعددة، وإنتاج رسوم بيانية احترافية.

### هيكلية المشروع

```
sales_analysis/
├── pyproject.toml
├── data/
│   └── sales_data.csv
├── output/              (سيُنشأ تلقائيًا للرسوم البيانية)
└── src/
    └── sales_analysis/
        ├── __init__.py
        ├── data_loader.py
        ├── analysis.py
        ├── visualization.py
        └── main.py
```

```bash
uv init sales_analysis
cd sales_analysis
uv add pandas numpy matplotlib
mkdir -p src/sales_analysis data output
touch src/sales_analysis/__init__.py
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

### الخطوة 1: توليد بيانات تجريبية واقعية

في مشروع حقيقي، ستقرأ ملف CSV جاهز لديك. هنا، سنولّد بيانات واقعية عشوائية لنتدرب عليها — ضعها في `data/generate_data.py` (سكربت مساعد منفصل، نُشغّله مرة واحدة فقط):

In [ ]:
"""سكربت مساعد لتوليد بيانات مبيعات تجريبية واقعية."""

import numpy as np
import pandas as pd

np.random.seed(42)  # لضمان نتائج قابلة لإعادة الإنتاج

n_records = 1000

products = ["لابتوب", "ماوس", "لوحة مفاتيح", "شاشة", "سماعات", "طابعة"]
product_prices = {
    "لابتوب": 3500, "ماوس": 50, "لوحة مفاتيح": 120,
    "شاشة": 900, "سماعات": 150, "طابعة": 600,
}
cities = ["الرياض", "جدة", "الدمام", "مكة", "المدينة"]

data = {
    "order_id": range(1, n_records + 1),
    "التاريخ": pd.date_range("2024-01-01", "2024-12-31", periods=n_records),
    "المنتج": np.random.choice(products, n_records),
    "المدينة": np.random.choice(cities, n_records, p=[0.35, 0.25, 0.15, 0.15, 0.10]),
    "الكمية": np.random.randint(1, 6, n_records),
}

df = pd.DataFrame(data)
df["سعر_الوحدة"] = df["المنتج"].map(product_prices)
df["الإجمالي"] = df["سعر_الوحدة"] * df["الكمية"]

# إدخال بعض القيم المفقودة عمدًا (كما يحدث في البيانات الحقيقية)
missing_indices = np.random.choice(df.index, size=20, replace=False)
df.loc[missing_indices, "المدينة"] = None

df.to_csv("data/sales_data.csv", index=False, encoding="utf-8-sig")
print(f"تم توليد {n_records} سجل مبيعات في data/sales_data.csv")

```bash
uv run python data/generate_data.py
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

### الخطوة 2: `data_loader.py` — تحميل وتنظيف البيانات

```python
"""وحدة تحميل وتنظيف بيانات المبيعات."""

from pathlib import Path

import pandas as pd

def load_sales_data(file_path: str | Path) -> pd.DataFrame:
    """تحميل بيانات المبيعات من ملف CSV.

    Args:
        file_path: مسار ملف CSV.

    Returns:
        DataFrame يحتوي بيانات المبيعات الخام.
    """
    df = pd.read_csv(file_path, encoding="utf-8-sig", parse_dates=["التاريخ"])
    return df

def clean_sales_data(df: pd.DataFrame) -> pd.DataFrame:
    """تنظيف بيانات المبيعات: معالجة القيم المفقودة والتكرارات.

    Args:
        df: DataFrame الخام.

    Returns:
        DataFrame نظيف جاهز للتحليل.
    """
    clean_df = df.copy()

    # تعويض المدن المفقودة بقيمة واضحة بدلاً من حذف الصفوف (نحتفظ بالبيانات)
    clean_df["المدينة"] = clean_df["المدينة"].fillna("غير محدد")

    # حذف أي صفوف مكررة بالكامل إن وُجدت
    clean_df = clean_df.drop_duplicates()

    # إضافة أعمدة مشتقة مفيدة للتحليل
    clean_df["الشهر"] = clean_df["التاريخ"].dt.month
    clean_df["اسم_الشهر"] = clean_df["التاريخ"].dt.month_name()
    clean_df["ربع_السنة"] = clean_df["التاريخ"].dt.quarter

    return clean_df
```

🧾 **للقراءة فقط** — محتوى ملف داخل المشروع؛ يُكتب على القرص ثم يُشغَّل من الطرفية.

### الخطوة 3: `analysis.py` — دوال التحليل

```python
"""وحدة تحليل بيانات المبيعات."""

import pandas as pd

def total_revenue(df: pd.DataFrame) -> float:
    """حساب إجمالي الإيرادات."""
    return float(df["الإجمالي"].sum())

def revenue_by_product(df: pd.DataFrame) -> pd.Series:
    """إجمالي الإيرادات لكل منتج، مرتبة تنازليًا."""
    return df.groupby("المنتج")["الإجمالي"].sum().sort_values(ascending=False)

def revenue_by_city(df: pd.DataFrame) -> pd.Series:
    """إجمالي الإيرادات لكل مدينة، مرتبة تنازليًا."""
    return df.groupby("المدينة")["الإجمالي"].sum().sort_values(ascending=False)

def monthly_trend(df: pd.DataFrame) -> pd.Series:
    """اتجاه الإيرادات الشهري عبر العام."""
    return df.groupby("الشهر")["الإجمالي"].sum().sort_index()

def top_n_products_by_quantity(df: pd.DataFrame, n: int = 5) -> pd.Series:
    """أكثر n منتجات مبيعًا من حيث الكمية."""
    return df.groupby("المنتج")["الكمية"].sum().sort_values(ascending=False).head(n)

def average_order_value(df: pd.DataFrame) -> float:
    """متوسط قيمة الطلب الواحد."""
    return float(df["الإجمالي"].mean())

def city_product_matrix(df: pd.DataFrame) -> pd.DataFrame:
    """جدول محوري: الإيرادات لكل منتج في كل مدينة."""
    return df.pivot_table(
        values="الإجمالي",
        index="المدينة",
        columns="المنتج",
        aggfunc="sum",
        fill_value=0,
    )

def generate_summary_report(df: pd.DataFrame) -> dict[str, float | int]:
    """توليد ملخص إحصائي شامل للبيانات."""
    return {
        "إجمالي_الإيرادات": total_revenue(df),
        "عدد_الطلبات": len(df),
        "متوسط_قيمة_الطلب": average_order_value(df),
        "أعلى_منتج_مبيعًا": str(revenue_by_product(df).index[0]),
        "أعلى_مدينة_مبيعًا": str(revenue_by_city(df).index[0]),
    }
```

🧾 **للقراءة فقط** — محتوى ملف داخل المشروع؛ يُكتب على القرص ثم يُشغَّل من الطرفية.

> **💡 نصيحة خبير:** لاحظ أن كل دالة تحليل تأخذ `DataFrame` وتُرجع نتيجة محددة — هذا نمط تصميم ممتاز (دوال نقية Pure Functions كما تعلمنا في الفصل 10) يجعل الكود **قابلاً للاختبار** بسهولة (سنتعلم `pytest` في الفصل 25) وقابلاً لإعادة الاستخدام على أي بيانات مشابهة.

### الخطوة 4: `visualization.py` — الرسوم البيانية

```python
"""وحدة توليد الرسوم البيانية لتحليل المبيعات."""

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

OUTPUT_DIR = Path("output")

def _ensure_output_dir() -> None:
    """التأكد من وجود مجلد الإخراج."""
    OUTPUT_DIR.mkdir(exist_ok=True)

def plot_revenue_by_product(revenue: pd.Series) -> None:
    """رسم بياني شريطي لإيرادات المنتجات."""
    _ensure_output_dir()
    plt.figure(figsize=(10, 6))
    revenue.plot(kind="bar", color="steelblue", edgecolor="black")
    plt.title("Revenue by Product", fontsize=14, fontweight="bold")
    plt.xlabel("Product")
    plt.ylabel("Revenue (SAR)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "revenue_by_product.png", dpi=200)
    plt.close()

def plot_revenue_by_city_pie(revenue: pd.Series) -> None:
    """رسم دائري لتوزيع الإيرادات حسب المدينة."""
    _ensure_output_dir()
    plt.figure(figsize=(8, 8))
    plt.pie(revenue.values, labels=revenue.index, autopct="%1.1f%%", startangle=90)
    plt.title("Revenue Distribution by City", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "revenue_by_city.png", dpi=200)
    plt.close()

def plot_monthly_trend(trend: pd.Series) -> None:
    """رسم خطي لاتجاه المبيعات الشهري."""
    _ensure_output_dir()
    plt.figure(figsize=(10, 6))
    plt.plot(trend.index, trend.values, marker="o", linewidth=2, color="darkgreen")
    plt.title("Monthly Revenue Trend", fontsize=14, fontweight="bold")
    plt.xlabel("Month")
    plt.ylabel("Revenue (SAR)")
    plt.grid(True, alpha=0.3)
    plt.xticks(range(1, 13))
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "monthly_trend.png", dpi=200)
    plt.close()

def plot_city_product_heatmap(matrix: pd.DataFrame) -> None:
    """رسم Heatmap لمصفوفة المدينة/المنتج."""
    _ensure_output_dir()
    plt.figure(figsize=(10, 6))
    plt.imshow(matrix.values, cmap="YlOrRd", aspect="auto")
    plt.colorbar(label="Revenue (SAR)")
    plt.xticks(range(len(matrix.columns)), matrix.columns, rotation=45)
    plt.yticks(range(len(matrix.index)), matrix.index)
    plt.title("Revenue Heatmap: City vs Product", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "city_product_heatmap.png", dpi=200)
    plt.close()
```

🧾 **للقراءة فقط** — محتوى ملف داخل المشروع؛ يُكتب على القرص ثم يُشغَّل من الطرفية.

> **💡 نصيحة خبير:** استخدمنا `plt.close()` بعد كل `savefig()` بدلاً من `plt.show()`. هذا ضروري في السكربتات التي تنتج عدة رسوم متتالية، لأنه يُحرر الذاكرة ويمنع تراكب الرسوم فوق بعضها بالخطأ.

### الخطوة 5: `main.py` — تجميع كل شيء

```python
"""نقطة الدخول الرئيسية لتحليل بيانات المبيعات."""

from sales_analysis.analysis import (
    city_product_matrix,
    generate_summary_report,
    monthly_trend,
    revenue_by_city,
    revenue_by_product,
    top_n_products_by_quantity,
)
from sales_analysis.data_loader import clean_sales_data, load_sales_data
from sales_analysis.visualization import (
    plot_city_product_heatmap,
    plot_monthly_trend,
    plot_revenue_by_city_pie,
    plot_revenue_by_product,
)

def print_report(summary: dict) -> None:
    """طباعة تقرير نصي منسّق في الطرفية."""
    print("=" * 50)
    print(" " * 15 + "تقرير تحليل المبيعات")
    print("=" * 50)
    for key, value in summary.items():
        label = key.replace("_", " ")
        if isinstance(value, float):
            print(f"{label}: {value:,.2f}")
        else:
            print(f"{label}: {value}")
    print("=" * 50)

def main() -> None:
    """تشغيل خط التحليل الكامل."""
    print("📊 تحميل البيانات...")
    raw_df = load_sales_data("data/sales_data.csv")

    print("🧹 تنظيف البيانات...")
    df = clean_sales_data(raw_df)
    print(f"   تم تحميل {len(df)} سجل بنجاح.\n")

    print("📈 تحليل البيانات...")
    summary = generate_summary_report(df)
    print_report(summary)

    print("\n🏆 أفضل 5 منتجات من حيث الكمية المباعة:")
    print(top_n_products_by_quantity(df, 5))

    print("\n🎨 توليد الرسوم البيانية...")
    plot_revenue_by_product(revenue_by_product(df))
    plot_revenue_by_city_pie(revenue_by_city(df))
    plot_monthly_trend(monthly_trend(df))
    plot_city_product_heatmap(city_product_matrix(df))

    print("✅ تم حفظ كل الرسوم البيانية في مجلد output/")

if __name__ == "__main__":
    main()
```

🧾 **للقراءة فقط** — محتوى ملف داخل المشروع؛ يُكتب على القرص ثم يُشغَّل من الطرفية.

تشغيل المشروع:

```bash
uv run python -m sales_analysis.main
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

الناتج المتوقع في الطرفية:

```
📊 تحميل البيانات...
🧹 تنظيف البيانات...
   تم تحميل 1000 سجل بنجاح.

==================================================
               تقرير تحليل المبيعات
==================================================
إجمالي الإيرادات: 1,245,780.00
عدد الطلبات: 1000
متوسط قيمة الطلب: 1,245.78
أعلى منتج مبيعًا: لابتوب
أعلى مدينة مبيعًا: الرياض
==================================================

🏆 أفضل 5 منتجات من حيث الكمية المباعة:
المنتج
ماوس          312
لوحة مفاتيح   298
...

🎨 توليد الرسوم البيانية...
✅ تم حفظ كل الرسوم البيانية في مجلد output/
```

ستجد الآن في مجلد `output/` أربع صور PNG احترافية جاهزة لوضعها في أي تقرير أو عرض تقديمي!

<details>
<summary>✅ الحل الكامل للمشروع (جميع الملفات مجتمعة)</summary>

**`pyproject.toml`** (مُولّد تلقائيًا عبر `uv init` + `uv add`، لا حاجة لتعديله يدويًا).

**`src/sales_analysis/data_loader.py`**، **`analysis.py`**، **`visualization.py`**، **`main.py`**: كما وردت بالكامل أعلاه دون أي تعديل — هذه هي الحل النموذجي الكامل.

**للتحقق من عمل كل شيء معًا:**

```bash
uv run python data/generate_data.py   # توليد البيانات (مرة واحدة)
uv run python -m sales_analysis.main  # تشغيل التحليل الكامل
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

**نقاط تصميم مهمة في هذا الحل:**
1. فصل الاهتمامات (Separation of Concerns): تحميل البيانات ≠ تحليلها ≠ رسمها ≠ عرضها — كل مسؤولية في ملف منفصل.
2. كل دالة تحليل هي دالة نقية (Pure Function): تأخذ DataFrame وتُرجع نتيجة، دون آثار جانبية.
3. استخدام `Path` بدلاً من نصوص خام للمسارات (كما تعلمنا في الفصل 16).
4. Type Hints في كل الدوال (كما تعلمنا في الفصل 15) — هذا يجعل الكود أوضح ويسمح لـ mypy باكتشاف الأخطاء.

</details>

## 20.7 أخطاء شائعة في تحليل البيانات

> **⚠️ تحذير 1: `SettingWithCopyWarning`**
> ```python
> subset = df[df[«السعر»] > 100]
> subset[«جديد»] = 1  # ⚠️ تحذير! قد لا يعمل كما تتوقع
> ```
> السبب: `subset` قد يكون «عرضًا» (view) وليس نسخة مستقلة من `df`. الحل:
> ```python
> subset = df[df[«السعر»] > 100].copy()
> subset[«جديد»] = 1  # ✅ آمن الآن
> ```

> **⚠️ تحذير 2: الخلط بين `.mean()` على عمود يحتوي نصوصًا**
> إذا حاولت حساب `.mean()` على عمود يحتوي قيمًا غير رقمية (نصوص مختلطة مع أرقام)، ستحصل على خطأ `TypeError`. تحقق دائمًا من `df.dtypes` قبل إجراء عمليات حسابية.

> **⚠️ تحذير 3: مقارنة القيم المفقودة بـ `==`**
> ```python
> df[df[«العمود»] == None]  # ❌ لا يعمل أبدًا! NaN != NaN رياضيًا
> df[df[«العمود»].isna()]   # ✅ الطريقة الصحيحة
> ```

> **💡 نصيحة خبير شاملة:** اجعل قاعدة «استكشف قبل أن تحلل» عادة ثابتة: عند كل بيانات جديدة، ابدأ دائمًا بـ `df.head()`, `df.info()`, `df.describe()`, `df.isna().sum()` قبل أي تحليل آخر. هذه العادة البسيطة تمنع 80% من الأخطاء الشائعة في تحليل البيانات.

## 20.8 تمارين

### 🟢 مستوى مبتدئ

1. أنشئ `DataFrame` يحتوي بيانات 5 طلاب (اسم، درجة في مادتين). احسب متوسط كل مادة باستخدام `.mean()`.
2. استخدم NumPy لإنشاء مصفوفة من 10 أرقام عشوائية بين 1 و100، ثم اطبع الأكبر والأصغر والمتوسط.
3. ارسم رسمًا بيانيًا شريطيًا بسيطًا يوضح عدد ساعات الدراسة لـ 5 أيام من الأسبوع.

<details>
<summary>✅ الحل المقترح</summary>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1
students = pd.DataFrame({
    "الاسم": ["أحمد", "سارة", "خالد", "منى", "يوسف"],
    "الرياضيات": [85, 92, 78, 95, 88],
    "العلوم": [90, 85, 82, 91, 79],
})
print(students["الرياضيات"].mean())
print(students["العلوم"].mean())

# 2
numbers = np.random.randint(1, 101, 10)
print(numbers)
print("الأكبر:", numbers.max())
print("الأصغر:", numbers.min())
print("المتوسط:", numbers.mean())

# 3
days = ["السبت", "الأحد", "الاثنين", "الثلاثاء", "الأربعاء"]
hours = [2, 3, 1.5, 4, 2.5]
plt.bar(days, hours, color="purple")
plt.title("ساعات الدراسة")
plt.show()

</details>

### 🟡 مستوى متوسط

4. لديك DataFrame يحتوي بيانات موظفين (اسم، قسم، راتب). استخدم `groupby` لحساب متوسط الراتب في كل قسم، ثم ارسم النتيجة كرسم شريطي.
5. أنشئ DataFrame يحتوي قيمًا مفقودة عمدًا في عمود «العمر»، ثم عوّضها بالوسيط (median) بدلاً من المتوسط، واشرح سبب اختيارك.

<details>
<summary>✅ الحل المقترح</summary>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 4
employees = pd.DataFrame({
    "اسم": ["علي", "فاطمة", "سعد", "ليلى", "عمر"],
    "القسم": ["مبيعات", "تقنية", "مبيعات", "تقنية", "مالية"],
    "الراتب": [8000, 12000, 8500, 13000, 9500],
})
avg_salary = employees.groupby("القسم")["الراتب"].mean()
print(avg_salary)
avg_salary.plot(kind="bar", color="teal")
plt.title("متوسط الراتب حسب القسم")
plt.ylabel("الراتب")
plt.tight_layout()
plt.show()

# 5
data = pd.DataFrame({"العمر": [25, 30, None, 45, 28, None, 90]})
# الوسيط أفضل من المتوسط هنا لأن القيمة 90 (قيمة متطرفة/Outlier)
# تؤثر بشدة على المتوسط لكن تأثيرها على الوسيط محدود جدًا
data["العمر"] = data["العمر"].fillna(data["العمر"].median())
print(data)

</details>

### 🔴 مستوى تحدي

6. وسّع مشروع تحليل المبيعات: أضف دالة جديدة `best_selling_month_per_product()` تحدد الشهر الأفضل مبيعًا لكل منتج على حدة (استخدم `groupby` بعمودين و`idxmax`). أضف أيضًا رسمًا بيانيًا جديدًا `plot_quarterly_comparison()` يقارن مبيعات الأرباع الأربعة للسنة بأعمدة متجاورة لكل مدينة (استخدم `pivot_table` ثم `.plot(kind="bar")`).

<details>
<summary>✅ الحل المقترح</summary>

```python
# إضافة إلى analysis.py
def best_selling_month_per_product(df: pd.DataFrame) -> pd.Series:
    """الشهر الأفضل مبيعًا لكل منتج."""
    grouped = df.groupby(["المنتج", "الشهر"])["الإجمالي"].sum()
    return grouped.groupby(level="المنتج").idxmax()

# إضافة إلى visualization.py
def plot_quarterly_comparison(df: pd.DataFrame) -> None:
    """مقارنة مبيعات الأرباع الأربعة لكل مدينة."""
    _ensure_output_dir()
    pivot = df.pivot_table(
        values="الإجمالي", index="ربع_السنة", columns="المدينة",
        aggfunc="sum", fill_value=0,
    )
    pivot.plot(kind="bar", figsize=(10, 6))
    plt.title("Quarterly Revenue Comparison by City")
    plt.xlabel("Quarter")
    plt.ylabel("Revenue (SAR)")
    plt.legend(title="City")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "quarterly_comparison.png", dpi=200)
    plt.close()
```

🧾 **للقراءة فقط** — مثال مقتطف يعتمد على `OUTPUT_DIR`، `_ensure_output_dir` من مثال سابق.

</details>

## 20.9 اختبر فهمك (Quiz)

<details>
<summary>1. ما الفرق الجوهري بين `list` العادية و `numpy.ndarray`؟</summary>

`ndarray` تتطلب أن تكون كل عناصرها من نفس النوع (موحدة)، وتدعم العمليات الحسابية مباشرة على كل العناصر دفعة واحدة (Vectorization) دون حلقات، وهي أسرع بكثير وأقل استهلاكًا للذاكرة لأنها منفذة بلغة C داخليًا. القوائم العادية مرنة (تقبل أنواعًا مختلطة) لكنها أبطأ في الحسابات الرقمية.
</details>

<details>
<summary>2. لماذا تفشل `df[(df["a"] > 5) and (df["b"] < 10)]` وما البديل الصحيح؟</summary>

لأن `and`/`or` في Python تتوقع قيمة Boolean مفردة، بينما الشرط `(df["a"] > 5)` يُرجع `Series` كاملة من قيم True/False وليس قيمة واحدة. البديل الصحيح هو استخدام `&` و`|` مع وضع أقواس حول كل شرط فرعي: `df[(df["a"] > 5) & (df["b"] < 10)]`.
</details>

<details>
<summary>3. متى تستخدم `loc` ومتى تستخدم `iloc`؟</summary>

`loc` يُستخدم للاختيار بناءً على **التسمية/الاسم** (label-based)، مثل اسم العمود أو قيمة الفهرس الفعلية. `iloc` يُستخدم للاختيار بناءً على **الموضع الرقمي** (position-based) تمامًا كالقوائم العادية (0، 1، 2...)، بغض النظر عن تسميات الفهرس.
</details>

<details>
<summary>4. ما وظيفة `groupby` ومتى تستخدمها؟</summary>

`groupby` يُستخدم لتقسيم البيانات إلى مجموعات بناءً على قيمة عمود (أو أكثر)، ثم تطبيق دالة تجميعية (sum, mean, count...) على كل مجموعة على حدة. يُستخدم للإجابة على أسئلة مثل «ما إجمالي المبيعات لكل مدينة؟» وهو مكافئ لـ `GROUP BY` في SQL.
</details>

<details>
<summary>5. لماذا نستخدم `.copy()` أحيانًا بعد تصفية DataFrame؟</summary>

لتجنب تحذير `SettingWithCopyWarning` والسلوك غير المتوقع الناتج عن أن الجزء المُصفّى قد يكون «عرضًا» (view) مرتبطًا بالبيانات الأصلية بدلاً من نسخة مستقلة. استخدام `.copy()` يضمن أننا نعمل على نسخة منفصلة تمامًا وآمنة للتعديل.
</details>

<details>
<summary>6. ما الفرق بين `plt.show()` و `plt.savefig()`؟</summary>

`plt.show()` يفتح نافذة تفاعلية لعرض الرسم على الشاشة مباشرة، بينما `plt.savefig()` يحفظ الرسم كملف صورة (PNG, JPG, PDF...) على القرص دون بالضرورة عرضه، وهذا مفيد جدًا في السكربتات الآلية التي تولّد عدة رسوم دون تدخل بشري.
</details>

## 20.10 ملخص الفصل (Cheat Sheet)

```python
# ===== NumPy =====
import numpy as np
arr = np.array([1, 2, 3])
arr.sum(); arr.mean(); arr.max(); arr.min(); arr.std()
filtered = arr[arr > 1]                      # Boolean Indexing

# ===== pandas: القراءة والاستكشاف =====
import pandas as pd
df = pd.read_csv("file.csv", encoding="utf-8")
df.head(); df.tail(); df.info(); df.describe()
df.shape; df.columns; df.dtypes
df.isna().sum()                              # القيم المفقودة

# ===== pandas: الاختيار والتصفية =====
df["col"]                                    # عمود -> Series
df[["col1", "col2"]]                         # أعمدة -> DataFrame
df.loc[0, "col"]                             # بالتسمية
df.iloc[0, 1]                                # بالموضع
df[(df["a"] > 5) & (df["b"] < 10)]           # تصفية بشروط (& | وليس and or)

# ===== pandas: التعديل =====
df["new_col"] = df["a"] * df["b"]
df = df.drop(columns=["col"])
df = df.rename(columns={"old": "new"})
df["col"] = df["col"].fillna(df["col"].mean())
df = df.dropna(subset=["col"])
df = df.sort_values("col", ascending=False)

# ===== pandas: التجميع والدمج =====
df.groupby("col")["val"].sum()
df.groupby("col")["val"].agg(["sum", "mean", "count"])
df.pivot_table(values="val", index="r", columns="c", aggfunc="sum")
pd.merge(df1, df2, on="key", how="left")
pd.concat([df1, df2], ignore_index=True)

# ===== pandas: التصدير =====
df.to_csv("out.csv", index=False, encoding="utf-8-sig")

# ===== Matplotlib =====
import matplotlib.pyplot as plt
plt.plot(x, y); plt.bar(x, y); plt.pie(values, labels=labels)
plt.scatter(x, y); plt.hist(data, bins=20)
plt.title(""); plt.xlabel(""); plt.ylabel(""); plt.legend()
plt.savefig("chart.png", dpi=200); plt.show(); plt.close()
```

🧾 **للقراءة فقط** — مثال مقتطف يعتمد على `df1`، `df2`، `labels` من مثال سابق.

| المهمة | الكود |
|---|---|
| تحميل CSV | `pd.read_csv(path, encoding="utf-8")` |
| فحص سريع | `df.head()`, `df.info()`, `df.describe()` |
| تصفية | `df[df["col"] > x]` |
| تجميع | `df.groupby("col")["val"].sum()` |
| جدول محوري | `df.pivot_table(...)` |
| رسم سريع | `df.plot(kind="bar")` |

## 20.11 ماذا بعد؟

لقد اكتسبت للتو قدرة قوية جدًا: تحويل بيانات خام فوضوية إلى رؤى وقرارات واضحة، مدعومة برسوم بيانية احترافية. هذه المهارة بالذات هي من أكثر المهارات طلبًا في سوق العمل التقني اليوم.

في الفصل القادم، سننتقل إلى مسار مختلف تمامًا لكنه بنفس الإثارة: **الأتمتة (Automation)**. سنتعلم كيف نجعل Python «يتصفح» الإنترنت نيابة عنا باستخدام مكتبتي `requests` و`BeautifulSoup`، لاستخراج بيانات من صفحات الويب تلقائيًا — مهارة تُعرف بـ **Web Scraping**، وتُستخدم في كل شيء من مراقبة الأسعار إلى جمع بيانات البحث.

---

### الخطوة التالية

- [الفصل الحادي والعشرون: مسار الأتمتة — استخراج البيانات من الويب (Web Scraping)](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/chapter-21.ipynb)
- [فهرس كل الدفاتر](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/00-index.ipynb)

*الفصل العشرون — من كتاب «بايثون 3.14: من الصفر إلى الاحتراف».*